# Pramaan · consented lip-sync test fake (LatentSync 1.5)

Makes a **realistic lip-synced test fake of our own consented clip** (Arnav, clip 1), so we can measure whether
Pramaan's Source-Match still finds the genuine original when the mouth has been redrawn.

**Rules we follow (RAKSHAM finale + CLAUDE.md):**
- Only footage and voices of team members who signed consent. **Never** a real public figure or anyone else.
- Output is a **test item**. It stays private, and every copy shown outside the eval carries a visible "AI-GENERATED TEST" label.
- Model: [LatentSync](https://github.com/bytedance/LatentSync) by ByteDance (Apache-2.0), checkpoint `ByteDance/LatentSync-1.5`. Listed in ACKNOWLEDGEMENTS.md.

**Before running:** Settings → Accelerator **GPU T4 x2** (or P100), Internet **On**, and the input dataset added (see the guide).
Then **Run All**. The first run takes roughly 15–30 minutes (estimate): ~8 min setup, the rest is generation.

In [ ]:
# 1 · Check the GPU
import subprocess, shutil
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout
      or "NO GPU FOUND: Settings -> Accelerator -> GPU T4 x2, then restart.")

In [ ]:
# 2 · Find our inputs in /kaggle/input
from pathlib import Path
inp = Path("/kaggle/input")
videos = sorted(p for p in inp.rglob("*") if p.suffix.lower() in (".mp4", ".mov"))
audios = sorted(p for p in inp.rglob("*") if p.suffix.lower() in (".wav", ".m4a", ".mp3", ".aac", ".ogg", ".opus"))
print("videos:", *videos, sep="\n  ")
print("audios:", *audios, sep="\n  ")
assert videos, "No video found. Add the 'pramaan-lipsync-input' dataset (right panel -> Add Input)."
VIDEO = next((v for v in videos if "clip1" in v.name), videos[0])
# Your own recorded voice wins over the robotic fallback. A phone recording saved as a video
# (e.g. scam_voice.mp4) counts too: only its sound is used.
voice = [p for p in audios + videos if p != VIDEO and "tts" not in p.name.lower()]
assert voice or audios, "No audio found in the dataset."
AUDIO = voice[0] if voice else audios[0]
print("\nUSING video:", VIDEO, "\nUSING audio:", AUDIO)

In [ ]:
%%bash
# 3 · Install LatentSync in its own Python 3.10 environment (exact versions from its requirements.txt)
set -e
cd /kaggle/working
[ -d LatentSync ] || git clone -q https://github.com/bytedance/LatentSync.git
cd LatentSync
git log -1 --format="LatentSync commit %h (%cd)"
pip -q install uv
# Kaggle's own Python is 3.13, too new for LatentSync's pinned torch 2.5.1. uv fetches a separate Python 3.10.
unset UV_SYSTEM_PYTHON UV_PYTHON
uv python install 3.10
PY=/kaggle/working/ls-env/bin/python
if ! $PY -c "import sys; assert sys.version_info[:2] == (3, 10)" 2>/dev/null; then
    rm -rf /kaggle/working/ls-env
    uv venv -q --python 3.10 --python-preference only-managed /kaggle/working/ls-env
fi
$PY --version
grep -v -i "^gradio" requirements.txt > req-nogradio.txt
uv pip install -q --python $PY --index-strategy unsafe-best-match -r req-nogradio.txt
(ldconfig -p | grep -q libGL.so.1) || (apt-get -qq update && apt-get -qq install -y libgl1 > /dev/null)
/kaggle/working/ls-env/bin/python -c "import torch; print('torch', torch.__version__, 'cuda ok:', torch.cuda.is_available())"

In [ ]:
%%bash
# 4 · Download the LatentSync 1.5 checkpoints (~5 GB; fits a 16 GB T4. Version 1.6 needs 18 GB, so we use 1.5)
set -e
cd /kaggle/working/LatentSync
/kaggle/working/ls-env/bin/python - <<'EOF'
from huggingface_hub import hf_hub_download
for f in ("latentsync_unet.pt", "whisper/tiny.pt"):
    p = hf_hub_download("ByteDance/LatentSync-1.5", f, local_dir="checkpoints")
    print("ok", p)
EOF
ls -la checkpoints checkpoints/whisper

In [ ]:
# 5 · Prepare inputs: 25 fps video, 16 kHz mono audio, audio trimmed to the video's length
import subprocess, json
W = Path("/kaggle/working")
def dur(p):
    o = subprocess.run(["ffprobe", "-v", "error", "-show_entries", "format=duration", "-of", "json", str(p)], capture_output=True, text=True).stdout
    return float(json.loads(o)["format"]["duration"])
vd, ad = dur(VIDEO), dur(AUDIO)
print(f"video {vd:.1f} s, audio {ad:.1f} s")
if ad > vd:
    print(f"NOTE: audio is longer than the video; the last {ad - vd:.1f} s of speech will be cut. Keep recordings under {vd:.0f} s.")
subprocess.run(["ffmpeg", "-v", "error", "-y", "-i", str(VIDEO), "-an", "-r", "25", "-c:v", "libx264", "-crf", "18", str(W / "in_video.mp4")], check=True)
subprocess.run(["ffmpeg", "-v", "error", "-y", "-i", str(AUDIO), "-t", f"{vd:.2f}", "-ac", "1", "-ar", "16000", str(W / "in_audio.wav")], check=True)
print("ready:", W / "in_video.mp4", W / "in_audio.wav")

In [ ]:
%%bash
# 6 · Generate the lip-synced test fake (the slow step)
set -e
# Kaggle points matplotlib at its notebook backend, which LatentSync's own Python can't load
export MPLBACKEND=Agg
cd /kaggle/working/LatentSync
time /kaggle/working/ls-env/bin/python -m scripts.inference \
    --unet_config_path "configs/unet/stage2.yaml" \
    --inference_ckpt_path "checkpoints/latentsync_unet.pt" \
    --inference_steps 20 \
    --guidance_scale 1.5 \
    --enable_deepcache \
    --video_path /kaggle/working/in_video.mp4 \
    --audio_path /kaggle/working/in_audio.wav \
    --video_out_path /kaggle/working/raw_out.mp4
ls -la /kaggle/working/raw_out.mp4

In [ ]:
# 7 · Two outputs:
#   fake_lipsync_clip1_arnav_en.mp4          -> unlabelled, PRIVATE, only for Pramaan's eval (a real scammer adds no label)
#   fake_lipsync_clip1_arnav_en_LABELLED.mp4 -> visible "AI-GENERATED TEST" label, for slides / stage / sharing
import subprocess
W = Path("/kaggle/working")
meta = ["-metadata", "comment=Synthetic test item made by Team Saksham with LatentSync from consented footage. Not a real statement."]
subprocess.run(["ffmpeg", "-v", "error", "-y", "-i", str(W / "raw_out.mp4"), "-c", "copy", *meta, str(W / "fake_lipsync_clip1_arnav_en.mp4")], check=True)
# Kaggle's ffmpeg can't draw text, so the label is drawn as an image with Pillow and laid over the video
from PIL import Image, ImageDraw, ImageFont
vw, vh = map(int, subprocess.run(["ffprobe", "-v", "error", "-select_streams", "v:0", "-show_entries", "stream=width,height",
                                  "-of", "csv=p=0", str(W / "raw_out.mp4")], capture_output=True, text=True).stdout.strip().split(","))
bar_h = max(40, vh // 14)
try:
    font = ImageFont.truetype("DejaVuSans-Bold.ttf", int(bar_h * 0.5))
except OSError:
    font = ImageFont.load_default(size=int(bar_h * 0.5))
bar = Image.new("RGBA", (vw, bar_h), (0, 0, 0, 160))
ImageDraw.Draw(bar).text((bar_h // 3, bar_h // 2), "AI-GENERATED TEST VIDEO - consented, not a real statement",
                         font=font, fill=(255, 255, 255, 255), anchor="lm")
bar.save(W / "label.png")
subprocess.run(["ffmpeg", "-v", "error", "-y", "-i", str(W / "raw_out.mp4"), "-i", str(W / "label.png"),
                "-filter_complex", "[0:v][1:v]overlay=0:H-h", "-c:a", "copy", *meta,
                str(W / "fake_lipsync_clip1_arnav_en_LABELLED.mp4")], check=True)
for f in ("fake_lipsync_clip1_arnav_en.mp4", "fake_lipsync_clip1_arnav_en_LABELLED.mp4"):
    print(f, round((W / f).stat().st_size / 1e6, 1), "MB")

In [ ]:
# 8 · Preview the labelled version here
from IPython.display import Video
Video("/kaggle/working/fake_lipsync_clip1_arnav_en_LABELLED.mp4", embed=True, width=640)

## Download
Right panel → **Output** (or *File → Download output* / the `/kaggle/working` folder) → download both
`fake_lipsync_clip1_arnav_en.mp4` and `fake_lipsync_clip1_arnav_en_LABELLED.mp4`.

Put them in `pramaan\data\media\test\` on the laptop. Claude then adds the unlabelled one to the **dev** split
(the frozen test split must not change) and measures Source-Match and the detectors on it.

Afterwards: **Stop session** (top right) so the GPU quota isn't used, and keep the Kaggle notebook and dataset **Private**.